In [16]:
import torch
import torch.nn as nn
import numpy as np

from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [18]:
TRAIN_PATH = "../data2/DATASET/train"
TEST_PATH = "../data2/DATASET/test"

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_CLASSES = 7

In [19]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [20]:
train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_PATH, transform=test_transform)

print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", train_dataset.classes)
print("Class to index:", train_dataset.class_to_idx)

Train images: 12271
Test images: 3068
Classes: ['1', '2', '3', '4', '5', '6', '7']
Class to index: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6}


In [21]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Train batches: 192
Validation batches: 48


In [22]:
targets = np.array(train_dataset.targets)
unique, counts = np.unique(targets, return_counts=True)

class_weights = 1.0 / torch.tensor(counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(unique)
class_weights = class_weights.to(device)

print("Class counts:", dict(zip(unique, counts)))
print("Class weights:", class_weights)

Class counts: {np.int64(0): np.int64(1290), np.int64(1): np.int64(281), np.int64(2): np.int64(717), np.int64(3): np.int64(4772), np.int64(4): np.int64(1982), np.int64(5): np.int64(705), np.int64(6): np.int64(2524)}
Class weights: tensor([0.6572, 3.0168, 1.1823, 0.1776, 0.4277, 1.2025, 0.3359])


In [23]:
model = models.resnet34(pretrained=True)

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, NUM_CLASSES)
)

model = model.to(device)

In [24]:
for param in model.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

In [25]:
best_val_acc = 0
patience = 7
patience_counter = 0

EPOCHS = 50

In [26]:
loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4  
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

In [27]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)

        _, predicted = torch.max(preds, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

        if i % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = correct / total

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            preds = model(batch_X)
            _, predicted = torch.max(preds, 1)

            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    val_acc = correct / total
    avg_loss = total_loss / len(train_loader.dataset)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.3f}, Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}")

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "emotion_model_rafdbv1.pth")
        print(f"New best model saved! Val Acc: {val_acc:.3f}")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

Epoch 1 | Batch 0/192 | Loss: 2.4427
Epoch 1 | Batch 50/192 | Loss: 1.9228
Epoch 1 | Batch 100/192 | Loss: 1.9413
Epoch 1 | Batch 150/192 | Loss: 1.8905
Epoch 1, Loss: 1.877, Train Acc: 0.419, Val Acc: 0.387
New best model saved! Val Acc: 0.387
Epoch 2 | Batch 0/192 | Loss: 1.9934
Epoch 2 | Batch 50/192 | Loss: 1.4750
Epoch 2 | Batch 100/192 | Loss: 1.5907
Epoch 2 | Batch 150/192 | Loss: 1.5327
Epoch 2, Loss: 1.590, Train Acc: 0.601, Val Acc: 0.670
New best model saved! Val Acc: 0.670
Epoch 3 | Batch 0/192 | Loss: 1.5899
Epoch 3 | Batch 50/192 | Loss: 1.3155
Epoch 3 | Batch 100/192 | Loss: 1.4587
Epoch 3 | Batch 150/192 | Loss: 1.7721
Epoch 3, Loss: 1.497, Train Acc: 0.652, Val Acc: 0.689
New best model saved! Val Acc: 0.689
Epoch 4 | Batch 0/192 | Loss: 1.4352
Epoch 4 | Batch 50/192 | Loss: 1.4106
Epoch 4 | Batch 100/192 | Loss: 1.3935
Epoch 4 | Batch 150/192 | Loss: 1.3031
Epoch 4, Loss: 1.422, Train Acc: 0.697, Val Acc: 0.688
Epoch 5 | Batch 0/192 | Loss: 1.4062
Epoch 5 | Batch 50/1

In [28]:
print(f"Best validation accuracy: {best_val_acc:.3f}")

Best validation accuracy: 0.823
